In [1]:
import os
import json
import pandas as pd
import numpy as np
import torch
from torch import nn

def save_3d_response_surfaces_csv(
    run_folder: str,
    fold: int,
    test_csv: str,
    sample_index: int,
    fluid_range: tuple[float, float],
    prop_range: tuple[float, float],
    n_fluid: int = 50,
    n_prop: int = 50
):
    """
    For each fluid type in `test_csv`, computes a grid of:
      Total.Fluid ∈ linspace(fluid_range)
      Total.Propellant ∈ linspace(prop_range)
    predicts BOE production and saves a CSV where:
      - columns = Total.Fluid
      - rows    = Total.Propellant
      - cell    = Predicted BOE
    The first cell (top-left) is labeled
      Total.Propellant ↓ / Total.Fluid →
    under <run_folder>/surfaces_csvs/.
    """
    # ─── Column names ─────────────────────────────────────────────────────────
    gpi_col           = "GPI (gross perforated interval ft)"
    prop_per_gpi_col  = "Proppant.per.GPI..lb.ft."
    fluid_per_gpi_col = "Fluid.per.GPI..gal.ft."
    total_prop_col    = "Total.Proppant.Volume"
    total_fluid_col   = "Total.Fluid"
    fluid_type_col    = "Fluid.Type"
    output1_col       = "BOE_Prodoction_2 year cum"

    # ─── Load hyperparams & norms ─────────────────────────────────────────────
    run_id = os.path.basename(os.path.normpath(run_folder))
    hp     = json.load(open(os.path.join(run_folder, f"{run_id}_hyperparams.json")))
    norms  = json.load(open(os.path.join(run_folder, f"{run_id}_norms.json")))

    layer_dims             = hp["layer_dims"]
    activations            = hp["activations"]
    include_ratio_features = hp.get("include_ratio_features", True)

    raw_mean = norms["y_mean"]
    raw_std  = norms["y_std"]
    y1_mean  = raw_mean[0] if isinstance(raw_mean, (list, tuple)) else raw_mean
    y1_std   = raw_std[0]  if isinstance(raw_std,  (list, tuple)) else raw_std

    x_mean = norms["x_mean"]
    x_std  = norms["x_std"]

    # ─── Load test data & pick sample ────────────────────────────────────────
    df     = pd.read_csv(test_csv)
    sample = df.iloc[sample_index]
    gpi    = float(sample[gpi_col])

    # ─── Build feature lists ─────────────────────────────────────────────────
    numeric_feats = list(x_mean.keys())
    if not include_ratio_features:
        numeric_feats = [
            c for c in numeric_feats
            if c not in (prop_per_gpi_col, fluid_per_gpi_col)
        ]
    fluid_types     = sorted(df[fluid_type_col].unique())
    dummy_feats_all = [f"{fluid_type_col}_{ft}" for ft in fluid_types]

    # ─── Define & load MLP ────────────────────────────────────────────────────
    class MLPNet(nn.Module):
        def __init__(self, in_dim, hidden_dims, activations, out_dim):
            super().__init__()
            layers, dims = [], [in_dim] + hidden_dims
            for i, h in enumerate(hidden_dims):
                layers.append(nn.Linear(dims[i], dims[i+1]))
                act = activations[i].lower()
                if   act == 'relu':     layers.append(nn.ReLU())
                elif act == 'tanh':     layers.append(nn.Tanh())
                elif act == 'sigmoid':  layers.append(nn.Sigmoid())
                elif act == 'softplus': layers.append(nn.Softplus())
                else:
                    raise ValueError(f"Unknown activation '{activations[i]}'")
            layers.append(nn.Linear(dims[-1], out_dim))
            self.net = nn.Sequential(*layers)

        def forward(self, x):
            return self.net(x)

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model  = MLPNet(
        in_dim      = len(numeric_feats) + len(dummy_feats_all),
        hidden_dims = layer_dims,
        activations = activations,
        out_dim     = 2
    ).to(device)

    model.load_state_dict(torch.load(
        os.path.join(run_folder, f"{run_id}_fold{fold}.pth"),
        map_location=device
    ))
    model.eval()

    # ─── Prepare grid ────────────────────────────────────────────────────────
    fluid_vals = np.linspace(fluid_range[0], fluid_range[1], n_fluid)
    prop_vals  = np.linspace(prop_range[0], prop_range[1], n_prop)

    # ─── Ensure output folder exists ─────────────────────────────────────────
    out_dir = os.path.join(run_folder, "surfaces_csvs")
    os.makedirs(out_dir, exist_ok=True)

    # ─── Loop per fluid type & write pivoted CSV ─────────────────────────────
    for ft in fluid_types:
        rows = []
        base = {f: float(sample[f]) for f in numeric_feats}
        for d in dummy_feats_all:
            base[d] = 1.0 if d == f"{fluid_type_col}_{ft}" else 0.0

        for p in prop_vals:
            for f in fluid_vals:
                base[total_prop_col]  = p
                base[total_fluid_col] = f
                if include_ratio_features:
                    base[prop_per_gpi_col]  = p / gpi
                    base[fluid_per_gpi_col] = f / gpi

                x_vec = [(base[c] - x_mean[c]) / x_std[c] for c in numeric_feats]
                x_vec += [base[d] for d in dummy_feats_all]
                X_in  = torch.tensor([x_vec], dtype=torch.float32).to(device)

                with torch.no_grad():
                    out = model(X_in).cpu().numpy().flatten()
                y1_pred = out[0] * y1_std + y1_mean

                rows.append({
                    total_prop_col:  p,
                    total_fluid_col: f,
                    output1_col:     float(y1_pred)
                })

        df_out = pd.DataFrame(rows)
        grid = df_out.pivot(
            index=total_prop_col,
            columns=total_fluid_col,
            values=output1_col
        )

        # save with a labelled top-left cell
        fname = f"{run_id}_fold{fold}_{ft}_surface.csv"
        grid.to_csv(
            os.path.join(out_dir, fname),
            index_label=f"Total.Propellant ↓ / Total.Fluid ->"
        )
        print(f"Saved {fname} (shape={grid.shape})")


In [2]:
fluid_range=(1e5, 1e7)
prop_range=(1e5, 1e7)
n_fluid=10
n_prop=10
sample_index=1

# relu

In [3]:
# ─── Example usage ─────────────────────────────────────────────────────────
save_3d_response_surfaces_csv(
    run_folder="/home/kamiar/chevron/Eagle-Ford/Second/d3feed88",
    fold=6,
    test_csv="/home/kamiar/chevron/Eagle-Ford/First/data/Eagle Ford Data(Eagle Ford)_test.csv",
    sample_index = sample_index,
    fluid_range=fluid_range,
    prop_range=prop_range,
    n_fluid=n_fluid,
    n_prop=n_prop,

)

Saved d3feed88_fold6_GEL_surface.csv (shape=(10, 10))
Saved d3feed88_fold6_SLKW_surface.csv (shape=(10, 10))
Saved d3feed88_fold6_XLINK_surface.csv (shape=(10, 10))
